# Практическое задание 3 Market

Работа со связанными таблицами клиентов, заказов, товаров и позиций заказов.

## Содержание

1. [Загрузка данных](#section-1)
2. [Создание EntitySet и связей](#section-2)
3. [Генерация признаков с помощью DFS](#section-3)

<a id="section-1"></a>

## 1. Загрузка данных

Загружаем четыре CSV из `data/raw` в отдельные DataFrame. Столбцы `join_date` и `order_date` преобразуем в даты.

| Таблица | Содержимое |
| --- | --- |
| `customers` | Клиенты: `client_id`, `name`, `join_date`, `region` |
| `orders` | Заказы: `order_id`, `client_id`, `order_date`, `total_amount` |
| `products` | Товары: `product_id`, `category`, `price` |
| `order_items` | Позиции заказов: `item_id`, `order_id`, `product_id`, `quantity` |

Ноутбук можно запускать из корня репозитория или каталога `notebooks`.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

customers = pd.read_csv(DATA_DIR / "customers.csv", parse_dates=["join_date"])
orders = pd.read_csv(DATA_DIR / "orders.csv", parse_dates=["order_date"])
products = pd.read_csv(DATA_DIR / "products.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv")

In [2]:
for name, table in {
    "customers": customers,
    "orders": orders,
    "products": products,
    "order_items": order_items,
}.items():
    print(f"{name}: {table.shape[0]} строк, {table.shape[1]} столбцов")
    display(table.head())

customers: 120 строк, 4 столбцов


,client_id,name,join_date,region
0,1,Alex Smith,2025-10-16,North
1,2,Morgan Smith,2024-04-24,South
2,3,Jordan Smith,2024-01-26,East
3,4,Taylor Smith,2024-10-08,West
4,5,Sam Smith,2024-09-07,Central


orders: 500 строк, 4 столбцов


,order_id,client_id,order_date,total_amount
0,1,1,2026-09-09,1788.55
1,2,2,2025-09-27,4917.84
2,3,3,2026-06-28,1068.63
3,4,4,2026-05-16,525.54
4,5,5,2025-11-11,2961.04


products: 40 строк, 3 столбцов


,product_id,category,price
0,1,Electronics,491.22
1,2,Clothing,370.88
2,3,Home,356.21
3,4,Books,175.18
4,5,Sports,492.55


order_items: 1468 строк, 4 столбцов


,item_id,order_id,product_id,quantity
0,1,1,1,2
1,2,1,19,4
2,3,1,28,1
3,4,2,2,2
4,5,2,33,2


<a id="section-2"></a>

## 2. Создание EntitySet и связей

Для используемой версии Featuretools нужны `pandas<3` и `setuptools<81` (Woodwork использует `pkg_resources`). Установка в окружение текущего ядра:

```python
%pip install featuretools==1.31.0 "pandas<3" "setuptools<81"
```

После изменения зависимостей перезапускаем ядро и выполняем ячейки с начала.

`EntitySet` библиотеки Featuretools хранит таблицы и связи между ними. Для каждой таблицы указываем уникальный ключ (`index`), а для клиентов и заказов также колонку времени (`time_index`).

Добавляем три связи «один ко многим»:

| Родительская таблица | Дочерняя таблица | Ключ связи |
| --- | --- | --- |
| `customers` | `orders` | `client_id` |
| `orders` | `order_items` | `order_id` |
| `products` | `order_items` | `product_id` |

У таблиц `products` и `order_items` нет собственной колонки времени, поэтому для них `time_index` не задаём.

In [3]:
import featuretools as ft


es = ft.EntitySet(id="market")

es = es.add_dataframe(
    dataframe_name="customers",
    dataframe=customers,
    index="client_id",
    time_index="join_date",
)
es = es.add_dataframe(
    dataframe_name="orders",
    dataframe=orders,
    index="order_id",
    time_index="order_date",
)
es = es.add_dataframe(
    dataframe_name="products",
    dataframe=products,
    index="product_id",
)
es = es.add_dataframe(
    dataframe_name="order_items",
    dataframe=order_items,
    index="item_id",
)

es = es.add_relationship(
    parent_dataframe_name="customers",
    parent_column_name="client_id",
    child_dataframe_name="orders",
    child_column_name="client_id",
)
es = es.add_relationship(
    parent_dataframe_name="orders",
    parent_column_name="order_id",
    child_dataframe_name="order_items",
    child_column_name="order_id",
)
es = es.add_relationship(
    parent_dataframe_name="products",
    parent_column_name="product_id",
    child_dataframe_name="order_items",
    child_column_name="product_id",
)

display(es)

/home/vaiforic/P/python/pmrd2/venv/lib/python3.14/site-packages/woodwork/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/vaiforic/P/python/pmrd2/venv/lib/python3.14/site-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/home/vaiforic/P/python/pmrd2/venv/lib/python3.14/site-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/home/vaiforic/P/python/pmrd2/venv/lib/python3.14/sit

Entityset: market
  DataFrames:
    customers [Rows: 120, Columns: 4]
    orders [Rows: 500, Columns: 4]
    products [Rows: 40, Columns: 3]
    order_items [Rows: 1468, Columns: 4]
  Relationships:
    orders.client_id -> customers.client_id
    order_items.order_id -> orders.order_id
    order_items.product_id -> products.product_id

<a id="section-3"></a>

## 3. Генерация признаков с помощью DFS

DFS (Deep Feature Synthesis) автоматически собирает характеристики клиентов из связанных таблиц. В результате получаем одну строку на клиента, а в столбцах — исходные и новые признаки.

Используем следующие примитивы:

| Тип | Примитивы | Назначение |
| --- | --- | --- |
| Агрегационные | `count`, `sum`, `mean`, `min`, `max`, `num_unique` | Количество связанных строк, сумма, среднее, минимум, максимум и число уникальных значений |
| Трансформационные | `year`, `month`, `weekday` | Год, месяц и день недели из даты |

`max_depth=2` разрешает до двух уровней построения признаков. Например, сначала считаем число позиций каждого заказа, затем среднее число позиций в заказах клиента. Примитивы применяются только к совместимым типам столбцов.

`feature_matrix` — готовая таблица признаков с индексом `client_id`, `feature_defs` — описания их вычисления.

Здесь анализируем всю доступную историю без `cutoff_time`. При прогнозировании на конкретную дату нужно ограничить историю этой датой, чтобы не использовать будущие данные.

In [4]:
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name="customers",
    agg_primitives=["count", "sum", "mean", "min", "max", "num_unique"],
    trans_primitives=["year", "month", "weekday"],
    max_depth=2,
)

print(f"Размер матрицы признаков: {feature_matrix.shape[0]} строк × {feature_matrix.shape[1]} столбцов")
print(f"Количество клиентов: {feature_matrix.shape[0]}")
print(f"Количество признаков: {feature_matrix.shape[1]}")
display(feature_matrix.head())

/home/vaiforic/P/python/pmrd2/venv/lib/python3.14/site-packages/featuretools/computational_backends/feature_set_calculator.py:785: FutureWarning: The provided callable <function mean at 0x7efea5b337f0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  ).agg(to_agg)
/home/vaiforic/P/python/pmrd2/venv/lib/python3.14/site-packages/featuretools/computational_backends/feature_set_calculator.py:785: FutureWarning: The provided callable <function max at 0x7efea5b32cf0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  ).agg(to_agg)
/home/vaiforic/P/python/pmrd2/venv/lib/python3.14/site-packages/featuretools/computational_backends/feature_set_calculator.py:785: FutureWarning: The provided callable <function sum at 0x7efea5b321f0> is currently using Ser

Размер матрицы признаков: 120 строк × 37 столбцов
Количество клиентов: 120
Количество признаков: 37


,region,COUNT(orders),MAX(orders.total_amount),MEAN(orders.total_amount),MIN(orders.total_amount),SUM(orders.total_amount),COUNT(order_items),MAX(order_items.quantity),MEAN(order_items.quantity),MIN(order_items.quantity),...,NUM_UNIQUE(orders.MONTH(order_date)),NUM_UNIQUE(orders.WEEKDAY(order_date)),NUM_UNIQUE(orders.YEAR(order_date)),SUM(orders.MAX(order_items.quantity)),SUM(orders.MEAN(order_items.quantity)),SUM(orders.MIN(order_items.quantity)),MAX(order_items.orders.total_amount),MEAN(order_items.orders.total_amount),MIN(order_items.orders.total_amount),SUM(order_items.orders.total_amount)
client_id,,,,,,,,,,,,,,,,,,,,,
32,South,4,4531.01,3104.0075,1827.75,12416.03,15,5.0,2.866667,1.0,...,3,2,1,17.0,12.950000,10.0,4531.01,3480.176667,1827.75,52202.65
3,East,5,3858.35,2692.8140,1068.63,13464.07,14,5.0,3.214286,1.0,...,4,4,2,22.0,16.533333,13.0,3858.35,3170.017857,1068.63,44380.25
21,North,5,3800.12,1269.7820,103.15,6348.91,10,5.0,1.600000,1.0,...,3,2,1,11.0,7.300000,6.0,3800.12,2264.738000,103.15,22647.38
15,Central,4,5195.44,2620.5800,215.66,10482.32,13,5.0,3.076923,1.0,...,4,3,2,15.0,11.500000,6.0,5195.44,3175.561538,215.66,41282.30
14,West,3,5575.51,2821.5100,586.88,8464.53,9,5.0,3.222222,1.0,...,2,2,2,11.0,10.250000,8.0,5575.51,3566.386667,586.88,32097.48


In [5]:
print("Полный список признаков:")
for number, feature in enumerate(feature_defs, start=1):
    print(f"{number}. {feature.get_name()}")

print("\nПримеры признаков глубины 2:")
for feature in [feature for feature in feature_defs if feature.get_depth() == 2][:5]:
    print(feature.get_name())

Полный список признаков:
1. region
2. COUNT(orders)
3. MAX(orders.total_amount)
4. MEAN(orders.total_amount)
5. MIN(orders.total_amount)
6. SUM(orders.total_amount)
7. COUNT(order_items)
8. MAX(order_items.quantity)
9. MEAN(order_items.quantity)
10. MIN(order_items.quantity)
11. SUM(order_items.quantity)
12. MONTH(join_date)
13. WEEKDAY(join_date)
14. YEAR(join_date)
15. MAX(orders.COUNT(order_items))
16. MAX(orders.MEAN(order_items.quantity))
17. MAX(orders.MIN(order_items.quantity))
18. MAX(orders.SUM(order_items.quantity))
19. MEAN(orders.COUNT(order_items))
20. MEAN(orders.MAX(order_items.quantity))
21. MEAN(orders.MEAN(order_items.quantity))
22. MEAN(orders.MIN(order_items.quantity))
23. MEAN(orders.SUM(order_items.quantity))
24. MIN(orders.COUNT(order_items))
25. MIN(orders.MAX(order_items.quantity))
26. MIN(orders.MEAN(order_items.quantity))
27. MIN(orders.SUM(order_items.quantity))
28. NUM_UNIQUE(orders.MONTH(order_date))
29. NUM_UNIQUE(orders.WEEKDAY(order_date))
30. NUM_UNIQU